In [1]:
import glob
import json
import gc
import os
import pandas as pd
import shutil
import numpy as np
from dotenv import load_dotenv
from huggingface_hub import login, snapshot_download, hf_hub_download, upload_file
from transformer_lens import HookedTransformer

from config import RunConfig
from utils import (
    build_classifiers,
    create_results_path,
    load_trained_clfs,
    save_clf_with_skops,
    save_dataset_locally,
    split_by_layer,
    upload_repo_to_hf,
    extract_model_activations,
safe_roc_auc, upload_repo_to_hf,
make_chat_settings
)
from sklearn.metrics import classification_report
from probes_dataset_creation_script import (
    create_canonical_dataset,
    distractor_word_data,
    neutral_filler_data,
    no_rule_keyword,
    opposite_statuses_rules,
    #canonical_test_heldout_split
)
from train_probes import training
from evaluate_probes import evaluate
from plot_probes import (
    accuracies_from_evaluation_results,
    plot_accuracy_per_layer,
    plot_auroc_curves,
)
from control_experiments import (
    distractor_control,
    double_rule_control,
    neutral_filler_control,
    no_keyword_control,
    p_value_control,
    train_on_shuffled_labels,
    weights_vs_diff_of_means,
)
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import classification_report
import skops.io as sio

In [2]:
login()

## Evaluate English Probe on other languages

In [3]:
repo_ix = "veerlosar/prism-model-activations"

In [4]:
df_path = hf_hub_download(repo_ix, "full_adherence_gpt_mini.parquet", repo_type="dataset")
fulldf = pd.read_parquet(df_path)
langs = fulldf["language"].unique().tolist()

In [5]:
#selected_langs = ['de', 'it', 'yo', 'hi', 'ig', 'ru', 'ur']
trusted_types = [
    "sklearn.linear_model._logistic.LogisticRegression",
    "numpy.ndarray",
    "numpy.dtype",
]

In [12]:
# loading classifiers
# taking LogReg at layer 24 - max roc
logreg_en_clf = sio.load(
    "results_adherence_gpt_en/trained_probes/LogisticRegression_layer_24_en.skops",
    trusted=trusted_types
)

In [13]:
logreg_en_clf

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",5000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

## Evaluating English probe on other languages

In [16]:
%%time
logreg24_evals = {}
for lang in selected_langs:
    # loading X
    logreg24_evals[lang] = {}
    print(f'Loading results for {lang}...')
    x_path = hf_hub_download(
        repo_ix,
        f"meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_mini_activations/X_adh_{lang}.npy",
        repo_type="dataset"
    )
    X = np.load(x_path)
    # loading y
    y_path = hf_hub_download(
        repo_ix,
        f"meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_mini_activations/y_adh_{lang}.npy",
        repo_type="dataset"
    )
    y = np.load(y_path)
    X_dict = split_by_layer(X)
    del X
    print(f'Loaded {lang}. Evaluating..')
    for layer in range(32):
        predictions = logreg_en_clf.predict(X_dict[layer])
        report = classification_report(y, predictions, output_dict=True)
        report["roc_auc"] = safe_roc_auc(y, logreg_en_clf.predict_proba(X_dict[layer])[:, 1])
        logreg24_evals[lang][layer] = report
    del X_dict
    del y
    gc.collect()
print('Saving...')
with open("results_adherence_gpt_en/eval_probes/EnLogReg24OnOtherLangs.json", "w") as file:
    json.dump(logreg24_evals, file, indent=4)

Loading results for de...
Loaded de. Evaluating..


/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Loading results for it...


meta-llama/Llama-3.1-8B-Instruct/adheren(…): reconstructing file:   0%|          |  0.00B / 7.36GB            

meta-llama/Llama-3.1-8B-Instruct/adheren(…): downloading bytes:           |  0.00B            

...

meta-llama/Llama-3.1-8B-Instruct/adheren(…): reconstructing file:   0%|          |  0.00B /  112kB            

meta-llama/Llama-3.1-8B-Instruct/adheren(…): downloading bytes:           |  0.00B            

Loaded ur. Evaluating..


/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/masha/Desktop/Projects/PRISM/prism-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Saving...
CPU times: user 7min 2s, sys: 2min 3s, total: 9min 5s
Wall time: 16min 23s


In [19]:
# visualisations are saved to vis_folder
logreg24_evals.keys(), logreg24_evals['de'][24]

(dict_keys(['de', 'it', 'yo', 'hi', 'ig', 'ru', 'ur']),
 {'0': {'precision': 0.40124740124740127,
   'recall': 0.5975232198142415,
   'f1-score': 0.48009950248756217,
   'support': 1938.0},
  '1': {'precision': 0.9300699300699301,
   'recall': 0.8572136836886465,
   'f1-score': 0.8921568627450981,
   'support': 12102.0},
  'accuracy': 0.8213675213675213,
  'macro avg': {'precision': 0.6656586656586657,
   'recall': 0.727368451751444,
   'f1-score': 0.6861281826163301,
   'support': 14040.0},
  'weighted avg': {'precision': 0.8570743416897264,
   'recall': 0.8213675213675213,
   'f1-score': 0.8352788595984382,
   'support': 14040.0},
  'roc_auc': 0.8032643582183023})

## Training on different languages

In [6]:
classifiers = [
    LogisticRegression(max_iter=5000, class_weight="balanced", C=0.1),
    CalibratedClassifierCV(
        estimator=SVC(kernel='rbf', degree=8, gamma='scale', cache_size=2000, random_state=42),
        cv=5,
        ensemble=False,
        n_jobs=-1,
    )
]

In [7]:
def split_3way(y, groups, seed=42):
    """train / val / test, stratified on label AND disjoint on groups."""
    idx = np.arange(len(y))
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trval, test = next(outer.split(idx, y, groups))
    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, val_rel = next(inner.split(trval, y[trval], groups[trval]))
    return trval[tr_rel], trval[val_rel], test

In [11]:
selected_langs = ['it', 'yo', 'hi', 'ig', 'ru', 'ur']
selected_langs

['it', 'yo', 'hi', 'ig', 'ru', 'ur']

In [12]:
%%time
for lang in selected_langs:

    # load the data
    print(f'Loading results for {lang}...')
    x_path = hf_hub_download(
        repo_ix,
        f"meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_mini_activations/X_adh_{lang}.npy",
        repo_type="dataset"
    )
    X = np.load(x_path)
    # loading y
    y_path = hf_hub_download(
        repo_ix,
        f"meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_mini_activations/y_adh_{lang}.npy",
        repo_type="dataset"
    )
    y = np.load(y_path)

    # split
    print('Splitting the results for', lang)
    langdf = fulldf.loc[fulldf["language"] == lang].copy()
    groups = np.array(
    [f"{r['rule_status']}|{r['topic']}|{r['category']}" for r in langdf.to_dict(orient='records')]
    )
    train_ids, valid_ids, test_ids = split_3way(y, groups)
    x_adh_train = X[train_ids]
    y_adh_train = y[train_ids]
    
    x_adh_valid = X[valid_ids]
    y_adh_valid = y[valid_ids]
    
    x_adh_test = X[test_ids]
    y_adh_test = y[test_ids]
    
    del X
    del y
    
    gc.collect()

    # training
    print('Training for', lang)
    run_cfg = RunConfig(
        language=lang,
        n_layers=32,
        dataset_name=f"adherence_gpt_{lang}",
        results_folder=f"results_adherence_gpt_{lang}/",
    )
    create_results_path(run_cfg)
    trained_classifiers = training(run_cfg, classifiers, split_by_layer(x_adh_train), y_adh_train)
    trained_probes_path = save_clf_with_skops(run_cfg, trained_classifiers)

    gc.collect()

    # evaluate
    print('Evaluating for', lang)
    adh_valid_X = split_by_layer(x_adh_valid)
    adh_validation_evals, adh_validation_eval_path = evaluate(
        run_cfg, trained_classifiers, adh_valid_X, y_adh_valid, save_path_prefix=f"{lang}AdherenceValid"
    )

    adh_test_X = split_by_layer(x_adh_test)
    adh_test_evals, adh_test_eval_path = evaluate(
        run_cfg, trained_classifiers, adh_test_X, y_adh_test, save_path_prefix=f"{lang}AdherenceTest"
    )
    del trained_classifiers
    del x_adh_valid
    del x_adh_train
    del x_adh_test
    del y_adh_train
    del y_adh_valid
    del y_adh_test
    del adh_valid_X
    del adh_test_X
    del adh_validation_evals
    del adh_test_evals
    gc.collect()
    print('Finished training for', lang)

    # uploading to HF
    print('Uploading to HF...')
    upload_repo_to_hf(
        f"results_adherence_gpt_{lang}/",
        run_cfg,
        repo_type="dataset",
        repo_id=repo_ix,
        path_in_repo=f"meta-llama/Llama-3.1-8B-Instruct/adherence_gpt_results_{lang}",  
    )
    print('--------------------')

Loading results for it...
Splitting the results for it
Training for it
Evaluating for it
Finished training for it
Uploading to HF...
--------------------
Loading results for yo...
Splitting the results for yo
Training for yo
Evaluating for yo
Finished training for yo
Uploading to HF...
--------------------
Loading results for hi...
Splitting the results for hi
Training for hi
Evaluating for hi
Finished training for hi
Uploading to HF...
--------------------
Loading results for ig...
Splitting the results for ig
Training for ig
Evaluating for ig
Finished training for ig
Uploading to HF...
--------------------
Loading results for ru...
Splitting the results for ru
Training for ru
Evaluating for ru
Finished training for ru
Uploading to HF...
--------------------
Loading results for ur...
Splitting the results for ur
Training for ur
Evaluating for ur
Finished training for ur
Uploading to HF...
--------------------
CPU times: user 7h 26min 1s, sys: 2min 8s, total: 7h 28min 10s
Wall time: 11